# Simple AE training example

This is a direct, inspectable training example for `input/GF/fingerprint0.txt`. It does not use the GUFU pipeline, checkpoint helpers, or custom training functions. The AE layer sequence is written out below. The training loop uses a PyTorch `DataLoader` with `shuffle=True`, optimizes a minibatch Frobenius norm, and prints metrics in the notebook output only.

In [1]:
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_path = Path("input/GF/fingerprint0.txt")
if not data_path.is_file():
    raise FileNotFoundError(f"Could not find {data_path}; run the notebook from the GUFU repository root.")

# Each nonblank line starts with x,y, followed by space-separated AP records.
# Each AP record is BSSID,RSSI,standard-deviation,count. Only BSSID and RSSI
# are used here. Coordinates are deliberately ignored by the autoencoder.
scan_measurements = []
for line_number, line in enumerate(data_path.read_text(encoding="utf-8").splitlines(), start=1):
    fields = line.split()
    if not fields:
        continue

    coordinate_text = fields[0]  # parsed only to make the file layout explicit
    measurements = {}
    for ap_record in fields[1:]:
        parts = ap_record.split(",")
        if len(parts) < 2:
            raise ValueError(f"Malformed AP record on line {line_number}: {ap_record!r}")
        bssid = parts[0].strip().replace(":", "").replace("-", "").upper()
        rssi_dbm = float(parts[1])
        # If an AP appears twice in one scan, keep the stronger (less negative) RSSI.
        measurements[bssid] = max(measurements.get(bssid, -120.0), rssi_dbm)
    scan_measurements.append(measurements)

# A fixed, sorted column order makes every scan use the same AP positions.
bssid_vocabulary = sorted({bssid for scan in scan_measurements for bssid in scan})
column_for_bssid = {bssid: column for column, bssid in enumerate(bssid_vocabulary)}

# Missing AP readings are filled with -120 dBm. Then use the paper's
# normalization: (RSSI + 120) / 120. Thus a missing slot becomes 0.
raw_rssi = torch.full((len(scan_measurements), len(bssid_vocabulary)), -120.0)
for scan_index, measurements in enumerate(scan_measurements):
    for bssid, rssi_dbm in measurements.items():
        raw_rssi[scan_index, column_for_bssid[bssid]] = rssi_dbm
X = (raw_rssi + 120.0) / 120.0

print(f"file: {data_path}")
print(f"scans: {X.shape[0]} | APs: {X.shape[1]} | matrix shape: {tuple(X.shape)}")
print(f"normalized input range: {X.min().item():.4f} to {X.max().item():.4f}")
print(f"first scan: {scan_measurements[0] and len(scan_measurements[0])} observed APs")

file: input/GF/fingerprint0.txt
scans: 392 | APs: 673 | matrix shape: (392, 673)
normalized input range: 0.0000 to 0.7304
first scan: 54 observed APs


## Model

The layer sequence below matches the local AE in `ae.py`, copied into this notebook so the model is visible and can be edited here. Its two dropout layers are active during training and disabled for evaluation. The model accepts `[batch, AP]` tensors and returns a vector whose width is `latent_dim`, plus a reconstruction with the same width as the AP vocabulary. Set `latent_dim` in the model cell to any positive integer.

In [2]:
class AE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.Linear(256, latent_dim),
            nn.Conv1d(1, 1, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(1, 1, kernel_size=3, padding=1),
            nn.Dropout(p=0.0),
            nn.Conv1d(1, 1, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(1, 1, kernel_size=11, padding=5),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(1, 1, kernel_size=11, padding=5),
            nn.ReLU(),
            nn.ConvTranspose1d(1, 1, kernel_size=3, padding=1),
            nn.Dropout(p=0.0),
            nn.ConvTranspose1d(1, 1, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(1, 1, kernel_size=3, padding=1),
            nn.Linear(latent_dim, 256),
            nn.Linear(256, input_dim),
        )

    def forward(self, x):
        # The convolutions operate on [batch, channel=1, latent/features].
        x = x.unsqueeze(1)
        z = self.encoder(x)
        reconstruction = self.decoder(z)
        return z.squeeze(1), reconstruction.squeeze(1)

latent_dim = 32  # Change this to any positive integer.
model = AE(input_dim=X.shape[1], latent_dim=latent_dim).to(device)
print(model)
print(f"device: {device}")

AE(
  (encoder): Sequential(
    (0): Linear(in_features=673, out_features=256, bias=True)
    (1): Linear(in_features=256, out_features=32, bias=True)
    (2): Conv1d(1, 1, kernel_size=(3,), stride=(1,), padding=(1,))
    (3): ReLU()
    (4): Conv1d(1, 1, kernel_size=(3,), stride=(1,), padding=(1,))
    (5): Dropout(p=0.0, inplace=False)
    (6): Conv1d(1, 1, kernel_size=(3,), stride=(1,), padding=(1,))
    (7): ReLU()
    (8): Conv1d(1, 1, kernel_size=(11,), stride=(1,), padding=(5,))
  )
  (decoder): Sequential(
    (0): ConvTranspose1d(1, 1, kernel_size=(11,), stride=(1,), padding=(5,))
    (1): ReLU()
    (2): ConvTranspose1d(1, 1, kernel_size=(3,), stride=(1,), padding=(1,))
    (3): Dropout(p=0.0, inplace=False)
    (4): ConvTranspose1d(1, 1, kernel_size=(3,), stride=(1,), padding=(1,))
    (5): ReLU()
    (6): ConvTranspose1d(1, 1, kernel_size=(3,), stride=(1,), padding=(1,))
    (7): Linear(in_features=32, out_features=256, bias=True)
    (8): Linear(in_features=256, out_featu

## Training

This is the full training loop: shuffled minibatches, Adam, Frobenius loss, backward pass, and one printed line per epoch. Nothing is saved to disk. The epoch loss shown is the mean of the minibatch Frobenius norms; RMSE is printed alongside it to make the scale easier to interpret.

In [4]:
batch_size = 32
epochs = 300
learning_rate = 0.0001

data_loader = DataLoader(
    TensorDataset(X),
    batch_size=batch_size,
    shuffle=True,
)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
    model.train()
    batch_frobenius_losses = []
    squared_error_sum = 0.0
    value_count = 0

    for (batch,) in data_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        _latent, reconstruction = model(batch)
        residual = reconstruction - batch
        loss = torch.linalg.vector_norm(residual)
        loss.backward()
        optimizer.step()

        batch_frobenius_losses.append(loss.item())
        squared_error_sum += residual.detach().square().sum().item()
        value_count += residual.numel()

    mean_batch_frobenius = sum(batch_frobenius_losses) / len(batch_frobenius_losses)
    rmse = (squared_error_sum / value_count) ** 0.5
    print(
        f"epoch {epoch + 1:3d}/{epochs} | "
        f"mean batch Frobenius={mean_batch_frobenius:.6f} | "
        f"RMSE={rmse:.6f}"
    )

epoch   1/300 | mean batch Frobenius=6.354558 | RMSE=0.141422
epoch   2/300 | mean batch Frobenius=6.250814 | RMSE=0.139113
epoch   3/300 | mean batch Frobenius=6.149585 | RMSE=0.136860
epoch   4/300 | mean batch Frobenius=6.050893 | RMSE=0.134664
epoch   5/300 | mean batch Frobenius=5.954691 | RMSE=0.132523
epoch   6/300 | mean batch Frobenius=5.860924 | RMSE=0.130436
epoch   7/300 | mean batch Frobenius=5.769533 | RMSE=0.128402
epoch   8/300 | mean batch Frobenius=5.680452 | RMSE=0.126420
epoch   9/300 | mean batch Frobenius=5.593605 | RMSE=0.124487
epoch  10/300 | mean batch Frobenius=5.508908 | RMSE=0.122602
epoch  11/300 | mean batch Frobenius=5.426280 | RMSE=0.120763
epoch  12/300 | mean batch Frobenius=5.345648 | RMSE=0.118969
epoch  13/300 | mean batch Frobenius=5.266935 | RMSE=0.117217
epoch  14/300 | mean batch Frobenius=5.190064 | RMSE=0.115506
epoch  15/300 | mean batch Frobenius=5.114956 | RMSE=0.113834
epoch  16/300 | mean batch Frobenius=5.041535 | RMSE=0.112200
epoch  1

In [ ]:
from copy import deepcopy

model.eval();

with torch.no_grad():
    _, output = model(X[0:1].to('cuda'))
    
output[output<1e-3]=0.0

print(X[0:1])
print(output)